- начальная вершина выбирается случайно
- все исходящие ребра равновероятны

In [1]:
import random
import re
import pandas as pd
from collections import defaultdict

FILE_PATH = "edge_final_probabilities.csv"

In [2]:
def parse_pair(value):
    if pd.isna(value):
        return None
    parts = re.split(r"\s*(?:→|->)\s*", str(value).strip())
    if len(parts) == 2 and all(parts):
        return parts[0], parts[1]
    return None

In [3]:
def load_graph(file_path):
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()
    required = {"agent_1", "agent_2", "meaning"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Не хватает столбцов: {missing}")
    graph = defaultdict(list)
    for _, row in df.iterrows():
        a1 = str(row["agent_1"]).strip()
        a2 = str(row["agent_2"]).strip()
        meaning = str(row["meaning"]).strip()
        if pd.isna(row["agent_1"]) or pd.isna(row["agent_2"]):
            continue
        if pd.isna(row["meaning"]) or not a1 or not a2 or not meaning:
            continue
        pair = parse_pair(row.get("best_pair", None))
        if pair:
            source, target = pair
        else:
            source, target = a1, a2
        graph[source].append({
            "target": target,
            "meaning": meaning,
        })
    return dict(graph)

In [4]:
def generate_chain(graph, min_words=10, max_words=15, max_steps=30):
    if not graph:
        return None
    current = random.choice(list(graph.keys()))
    tokens = current.split()
    for _ in range(max_steps):
        if len(tokens) >= max_words:
            break
        edges = graph.get(current, [])
        if not edges:
            break
        edge = random.choice(edges)
        tokens.extend(edge["meaning"].split())
        tokens.extend(edge["target"].split())
        current = edge["target"]
    if min_words <= len(tokens) <= max_words:
        result = " ".join(tokens)
        return result[0].upper() + result[1:] + "."
    return None

In [5]:
def generate_many(
    graph,
    n=100,
    min_words=10,
    max_words=15,
    max_attempts=5000,
):
    results = set()
    for _ in range(max_attempts):
        if len(results) >= n:
            break
        sentence = generate_chain(
            graph,
            min_words=min_words,
            max_words=max_words,
        )
        if sentence is not None:
            results.add(sentence)
    return sorted(results)

In [6]:
if __name__ == "__main__":
    graph = load_graph(FILE_PATH)
    print("Количество начальных узлов:", len(graph))
    sentences = generate_many(
        graph,
        n=10000,
        min_words=10,
        max_words=15,
    )
    print("Уникальных цепочек:", len(sentences))
    for sentence in sentences[:50]:
        print(sentence)
    pd.DataFrame(
        {"sentence": sentences}
    ).to_csv(
        "generated_sentences.csv",
        index=False,
        encoding="utf-8-sig",
    )

/tmp/ipykernel_1577/2656026295.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Количество начальных узлов: 1283
Уникальных цепочек: 1406
Автор видеть умирать почерпать жизнь суть попадаться дерево застилали ее.
Автор означать к мать быть пресмыкаться написать на род броситься старый ходить с пристань.
Агитатор быть это объявить свой написать на наш случаться обречь с заморенный.
Агитатор выйти слово не быть соответствовать рот не прийти право работать лед.
Агитатор сделать мир знать мочь бывать на свет позволить узнать повторять мелькать ничтожество.
Ад высекать из огонь считать жить уметь кончить иметь образовать внимание.
Ад считать жить уметь из такой знать склад быть мать оскорбить дело мыть пол.
Адать делаться стать уйти за этот страдать мучиться свой перечислить бессодержательный.
Акация распознать иметь осенить отвечать жить необходимость постараться уметь помочь дать в постель.
Анафема мешать огорченный неприятность утверждать право страдать мучиться прегрешение стать мать затрудняться быть по палец.
Ангел соскочить лошадь мешать огорченный неприятность п